# Challenge 4: Spooky Telepathy ★★☆

**The mission:** Use entangled qubits to do things that look like telepathy. Make pairs that always agree or always disagree. Then use entanglement to squeeze **two bits of information into one qubit**, a trick called **superdense coding**, and use it to send a text message.

**Quantum ideas:** entanglement, Bell states, phase, superdense coding.
**Best for:** physics fans and anyone curious about "spooky action at a distance."
**Time plan:** Missions 1 to 5 take about 90 minutes. Spend the rest on "Make it yours."

In [ ]:
# Using Google Colab? Delete the "#" at the start of the next line, then run this cell once.
# !pip install -q "qiskit[visualization]"

In [ ]:
# SETUP: run this cell first (click it, then press Shift+Enter).
# You don't need to read or change anything in this cell.

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import math, random, qiskit

_sampler = StatevectorSampler()

def _needs_measure(qc):
    if qc.num_clbits == 0:
        raise ValueError("Your circuit has no measurements yet. Add qc.measure_all() as the last step of your circuit.")

def run(qc, shots=1000):
    """Run a circuit many times. Returns counts, e.g. {'0': 507, '1': 493}."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_counts()

def run_once(qc):
    """Run a circuit one time. Returns a single result as a string, e.g. '101'."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=1).result()[0].join_data().get_bitstrings()[0]

def run_many(qc, shots):
    """Run a circuit `shots` times. Returns every result in order, e.g. ['0', '1', '1', ...]."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_bitstrings()

def probabilities(qc):
    """Exact chance of each result, with no randomness. Measurements at the end are ignored."""
    sv = Statevector(qc.remove_final_measurements(inplace=False))
    return {str(k): round(float(v), 4) for k, v in sv.probabilities_dict().items() if v > 1e-9}

def _close(got, want, tol=0.02):
    return all(abs(got.get(k, 0) - want.get(k, 0)) <= tol for k in set(got) | set(want))

def _report(ok, yes, hint):
    if ok:
        print("\u2705 " + yes)
    else:
        print("\u274c Not yet. Hint: " + hint)

def _check_probs(qc, want, yes, hint):
    try:
        ok = _close(probabilities(qc), want)
    except Exception:
        ok = False
    _report(ok, yes, hint)

# ---- checks for this notebook ----
def text_to_bits(text):
    """'Hi' -> '0100100001101001' (8 bits per character)."""
    return "".join(format(ord(ch), "08b") for ch in text)

def bits_to_text(bits):
    """The reverse of text_to_bits."""
    return "".join(chr(int(bits[i:i + 8], 2)) for i in range(0, len(bits) - 7, 8))

def check_mission_1(make_bell_pair):
    _check_probs(make_bell_pair(), {"00": 0.5, "11": 0.5}, "Mission 1 complete: a Bell pair.",
                 "qc.h(0), then qc.cx(0, 1).")

def check_mission_2(qc):
    _check_probs(qc, {"01": 0.5, "10": 0.5}, "Mission 2 complete: a pair that always disagrees.",
                 "one gate on qubit 0 that flips 0 and 1 will do it.")

def check_mission_3(superdense):
    try:
        ok = all(_close(probabilities(superdense(m)), {m: 1.0}) for m in ["00", "01", "10", "11"])
    except Exception:
        ok = False
    _report(ok, "Mission 3 complete: two bits sent with one qubit!",
            "Alice: X if message[0] is '1', Z if message[1] is '1'. Bob: qc.cx(0, 1), then qc.h(0).")

def check_mission_4(send_text):
    try:
        ok = send_text("Qubit!") == "Qubit!"
    except Exception:
        ok = False
    _report(ok, "Mission 4 complete: text message delivered by superdense coding.",
            "for each pair, add run_once(superdense(pair)) to `received`.")

def check_mission_5(qc):
    _check_probs(qc, {"000": 0.5, "111": 0.5}, "Mission 5 complete: three-way entanglement!",
                 "start like a Bell pair, then use one more CNOT to pull qubit 2 in.")

print(f"\u2705 Setup complete (Qiskit {qiskit.__version__}). You're ready to go!")

## Mission 1: Make a Bell pair

You made this in the warm-up. Now wrap it in a function so you can make fresh pairs whenever you need them. The function returns the circuit **without** measuring, so later missions can add more gates.

**TODO:** H on qubit 0, then a CNOT from qubit 0 to qubit 1.

In [ ]:
def make_bell_pair():
    qc = QuantumCircuit(2)
    # TODO: H on qubit 0, then CNOT from qubit 0 to qubit 1

    return qc

qc = make_bell_pair()
qc.measure_all()
print(run(qc))
qc.draw("mpl")

In [ ]:
check_mission_1(make_bell_pair)

## Mission 2: Twins and opposites

Your Bell pair always **agrees**: `00` or `11`. One extra gate on qubit 0 can make a pair that always **disagrees**: `01` or `10`.

**TODO:** Add one gate to qubit 0 so the results are only `01` and `10`.

In [ ]:
qc = make_bell_pair()
# TODO: add ONE gate to qubit 0

print("Odds:", probabilities(qc))

In [ ]:
check_mission_2(qc)

Now try the Z gate on a Bell pair. Run the cell below. The odds look **exactly** like a normal Bell pair. But look at the math printout of the state: there's a minus sign. The Z gate changed the hidden **phase** of the pair. You can't see phase by measuring directly, but the next mission uses it to carry information.

In [ ]:
plain = make_bell_pair()
with_z = make_bell_pair()
with_z.z(0)

print("Plain Bell pair odds: ", probabilities(plain))
print("Bell pair + Z odds:   ", probabilities(with_z))
display(Statevector(plain).draw("latex"))
display(Statevector(with_z).draw("latex"))

## Mission 3: Superdense coding

Alice and Bob each take one qubit of a Bell pair (Alice has qubit 0, Bob has qubit 1) and go their separate ways. Later, Alice wants to send Bob **2 bits**, but she may only send him **one qubit**. Normally one qubit carries one bit. Thanks to the shared Bell pair, it can carry two.

Alice encodes by touching only her own qubit:

| Message | Alice applies to qubit 0 |
| --- | --- |
| `00` | nothing |
| `10` | X |
| `01` | Z |
| `11` | X, then Z |

Then she sends her qubit to Bob. Bob decodes using both qubits: a CNOT from qubit 0 to qubit 1, then H on qubit 0, then he measures.

**TODO:** Fill in Alice's encoding and Bob's decoding. `message[0]` is the first character and `message[1]` is the second.

In [ ]:
def superdense(message):
    """Send a 2-bit message like '10' with one Bell pair. Returns the circuit."""
    qc = make_bell_pair()
    qc.barrier()          # a barrier is just a visual divider in the drawing

    # --- Alice encodes (she only touches qubit 0) ---
    # TODO: X if message[0] is "1". Then Z if message[1] is "1".

    qc.barrier()

    # --- Bob decodes ---
    # TODO: CNOT from qubit 0 to qubit 1, then H on qubit 0

    qc.measure_all()
    return qc

for msg in ["00", "01", "10", "11"]:
    print("Alice sends", msg, "-> Bob measures", run(superdense(msg)))
superdense("11").draw("mpl")

In [ ]:
check_mission_3(superdense)

## Mission 4: Send a text message

Every character is 8 bits, so a message is a long string of 0s and 1s. Send it two bits at a time with superdense coding.

**TODO:** For each 2-bit pair, run the superdense circuit once and add Bob's result to `received`.

In [ ]:
def send_text(text):
    bits = text_to_bits(text)        # e.g. 'Hi' -> '0100100001101001'
    received = ""
    for i in range(0, len(bits), 2):
        pair = bits[i:i + 2]
        # TODO: send `pair` with superdense coding and add Bob's result to `received`
        received += "00"
    return bits_to_text(received)

message = "Hello from QSA!"
print("Bob received:", send_text(message))
print(f"That's {len(message) * 8} bits sent using only {len(message) * 4} qubits.")

In [ ]:
check_mission_4(send_text)

## Mission 5: Three-way telepathy

Can you entangle **three** qubits so they all agree, giving only `000` or `111`? This is called a **GHZ state**.

**TODO:** Build it. Hint: start with a Bell pair on qubits 0 and 1, then think about how a CNOT could pull qubit 2 in.

In [ ]:
qc = QuantumCircuit(3)
# TODO: entangle all three qubits

qc.measure_all()
print(run(qc))
qc.draw("mpl")

In [ ]:
check_mission_5(qc)

## Make it yours

Pick an idea below or invent your own:

- **Is it faster than light?** Entanglement looks instant. Can Alice use it to send Bob a message without sending her qubit? Test it: for each of Alice's four encodings, look only at Bob's qubit. What does he see? Explain why this can't send messages. Great for the Best Quantum Explainer award.
- **Quantum teleportation.** Move the state of a qubit from Alice to Bob using a Bell pair. Hints: use 3 qubits. Prepare qubit 0 with `qc.ry(1.2, 0)`. Make a Bell pair on qubits 1 and 2. Alice does `cx(0, 1)` and `h(0)`. Then Bob fixes his qubit with `cx(1, 2)` and `cz(0, 2)`. Compare the Bloch spheres before and after with `plot_bloch_multivector`.
- **Telepathy magic show.** Design a stage trick for your demo: an audience member picks a 2-bit message (or a letter) and your "mind reader" receives it.
- **Bigger telepathy.** Entangle 5 or 10 qubits. Can you make a chain where neighbors always disagree?
- **Message stats.** For a long message, chart how many qubits superdense coding saves compared to sending one bit per qubit.

## Explain it

Answer in your own words and be ready to say them out loud.

1. What does it mean for two qubits to be entangled?
2. How can one qubit carry two bits in superdense coding?
3. Does entanglement let you send messages faster than light? Why or why not?

Our answers: